# Using EcoFOCIpy to process raw field data

## SKQ202515S (Arctic Cruise - Sikuliaq)

## BO Oxygen Data



In [1]:
import yaml
import glob
import pandas as pd
import os

import EcoFOCIpy.io.sbe_ctd_parser as sbe_ctd_parser #<- instrument specific
import EcoFOCIpy.io.ncCFsave as ncCFsave
import EcoFOCIpy.metaconfig.load_config as load_config

/Users/bell/src/ecofocipy/src/EcoFOCIpy/__init__.py:62: UserWarning: Could not import all EcoFOCIpy submodules: cannot import name 'instruments' from partially initialized module 'EcoFOCIpy' (most likely due to a circular import) (/Users/bell/src/ecofocipy/src/EcoFOCIpy/__init__.py)
  warnings.warn(f"Could not import all EcoFOCIpy submodules: {e}")


In [2]:
sample_data_dir = '/Users/bell/ecoraid/2025/CTDcasts/skq202515s/' #root path to cruise directory
ecofocipy_dir = '/Users/bell/Programs/EcoFOCIpy/'

In [3]:
###############################################################
# edit to point to {cruise sepcific} raw datafiles 
datafile = sample_data_dir+'rawconverted/' #<- point to cruise and process all files within
oxydatafile = sample_data_dir+'working/DiscreteOxygens/Oxygen Data SKQ25.csv' #<- point to cruise and process all files within
cruise_name = 'skq202515s' #no hyphens
cruise_meta_file = sample_data_dir+'logs/SKQ202515S.yaml'
inst_meta_file = sample_data_dir+'logs/FOCI_standard_CTDpOxy.yaml' #<- copy to each deployment for simplicity?
group_meta_file = ecofocipy_dir+'staticdata/institutional_meta_example.yaml'
###############################################################

#init and load data
cruise = sbe_ctd_parser.sbe_btl()
filename_list = sorted(glob.glob(datafile + '*.btl'))

cruise_data = cruise.manual_parse(filename_list)

Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd001.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd002.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd003.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd004.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd005.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd006.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd007.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd008.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd009.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd010.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd011.btl
Processing /Users/bell/ecoraid/2025/CTDcasts/skq202515s/rawconverted/ctd012.btl
Processing /Users/bell/ecoraid/2025/CTDc

## Load csv Oxygen File

In [6]:
oxy_data = pd.read_csv(oxydatafile,delimiter='\t')
oxy_data

,Cast,Unnamed: 1,Niskin,Unnamed: 3,"O2, uM/l"
0,64,NaN,11,NaN,428.717383
1,63,NaN,3,NaN,282.671692
2,61,NaN,3,NaN,288.897492
3,65,NaN,8,NaN,279.585302
4,65,NaN,24,NaN,392.571796
5,65,NaN,6,NaN,302.355343
6,65,NaN,19,NaN,412.512317
7,65,NaN,2,NaN,309.158119
8,65,NaN,13,NaN,316.620042
9,65,NaN,4,NaN,307.329202


In [7]:
oxy_data.dropna(how='all', inplace=True)

## Merge Bottle and Nutrient Data but drop non nutrient vars?

In [8]:
keep_param = ['bottle','prdm']

for cast,cdata in cruise_data.items():
    try:
        matchcast = int((cast.split('.')[0]).lower().split('ctd')[-1])
        cruise_data[cast] = pd.merge(oxy_data[oxy_data['Cast']==matchcast],cdata.reset_index()[keep_param],right_on='bottle',left_on='Niskin').set_index('bottle').drop(columns=['Cast'])
        cruise_data[cast] = cruise_data[cast].dropna(how='all', axis=1)
    except:
        continue

## Add Deployment meta information

In [9]:
#just a dictionary of dictionaries - simple
with open(cruise_meta_file) as file:
    cruise_config = yaml.full_load(file)

## Add Instrument meta information

Time, depth, lat, lon should be added regardless (always our coordinates) but for a mooring site its going to be a (1,1,1,t) dataset
The variables of interest should be read from the data file and matched to a key for naming.  That key is in the inst_config file seen below and should represent common conversion names in the raw data

In [10]:
with open(inst_meta_file) as file:
    inst_oxy_config = yaml.full_load(file)

## Add institutional meta-information


In [11]:
with open(group_meta_file) as file:
    group_config = yaml.full_load(file)

## Save CF Netcdf files

Currently stick to netcdf3 classic... but migrating to netcdf4 (default) may be no problems for most modern purposes.  Its easy enough to pass the `format` kwargs through to the netcdf api of xarray.

In [12]:
#loop over all casts and perform tasks shown above

for cast in cruise_data.keys():
    try:
        cruise_data[cast] = cruise_data[cast].rename(columns={
                            'O2 (uM)':'O2',
                            'O2, uM/l':'O2',
                            'Oxygen (uM)':'O2',
                            'Niskin':'BTLID',
                            'prdm':'pressure',
                            'empty':'empty', #this will be ignored
                            'flag':'flag'})

        cruise_data_nc = ncCFsave.EcoFOCI_CFnc(df=cruise_data[cast], 
                                    instrument_yaml=inst_oxy_config, 
                                    operation_yaml=cruise_config,
                                    operation_type='ctd')

        cruise_data_nc.expand_dimensions(dim_names=['latitude','longitude','time'],geophys_sort=False)

        cruise_data_nc.variable_meta_data(variable_keys=list(cruise_data[cast].columns.values),drop_missing=False)
        #adding dimension meta needs to come after updating the dimension values... BUG?
        cruise_data_nc.dimension_meta_data(variable_keys=['time','latitude','longitude'])
        cruise_data_nc.temporal_geospatioal_meta_data_ctd(positiveE=False,conscastno=cast.split('.')[0])

        #add global attributes
        cruise_data_nc.deployment_meta_add(conscastno=cast.split('.')[0].upper())

        #add instituitonal global attributes
        cruise_data_nc.institution_meta_add(group_config)

        #add creation date/time - provenance data
        cruise_data_nc.provinance_meta_add()

        #provide intial qc status field
        cruise_data_nc.qc_status(qc_status='excellent') #<- options are unknown, excellent, probably good, mixed, unqcd

        cruise_data_nc.xarray2netcdf_save(xdf = cruise_data_nc.get_xdf(),
                                   filename=cruise_name+'c'+cast.lower().split('d')[-1].split('.')[0].zfill(3)+'_oxy.nc',format="NETCDF3_CLASSIC")
    except KeyError:
        print(f'Skipping {cast}')
    except RuntimeError:
        print(f'Skipping & Removing {cast}')
        os.remove(path=cruise_name+'c'+cast.lower().split('d')[-1].split('.')[0].zfill(3)+'_oxy.nc')

Skipping ctd046.btl


## Next Steps

QC of data (plot parameters with other instruments)
- be sure to updated the qc_status and the history